# Réplica de Toprani y Madisetti · AWS CloudFormation (con Google Gemini)

Este cuaderno analiza plantillas estáticamente. Genera hallazgos JSON con evidencia, un informe Markdown y, en modo de evaluación, una puntuación frente a la referencia reservada.

**Primera ejecución.** Selecciona el entorno de CPU y ejecuta las celdas en orden con `MODO = "simular"`. Luego usa `MODO = "detectar"`, `FASE = "smoke"` e `ID_CORRIDA = "smoke-gemini"` para la primera prueba real. Colab debe tener acceso al Secret `GEMINI_API_KEY` de tu cuenta de Google Cloud / Google AI Studio.

**Evaluación.** El archivo `Referencias_Toprani_Evaluador.zip` se comparte solo con la persona que puntúa. Súbelo a su carpeta privada de Drive. El equipo de detección comparte la carpeta de corridas con el evaluador. No se incluyen etiquetas en el cuaderno.

[Artículo](https://ieeexplore.ieee.org/document/10965635) · [Documentación de Gemini](https://ai.google.dev/gemini-api/docs)

## 1. Entorno y carpeta del estudio

`simular` usa un backend local de prueba. `detectar` invoca directamente las APIs de Google Gemini (`gemini-3.8-flash` y `gemini-embedding-2`). `evaluar` usa archivos guardados y no consume APIs. Conserva el mismo ID para reanudar. Cambia el ID para realizar una corrida nueva. Trabaja con una sesión escritora por ID.

In [76]:
from pathlib import Path
from google.colab import drive, userdata
import base64, hashlib, importlib, io, subprocess, sys, zipfile

MODO = 'evaluar'  # @param ['simular', 'detectar', 'evaluar']
FASE = 'formal'  # @param ['smoke', 'piloto', 'formal', 'actualizacion']
ID_CORRIDA = 'formal-gemini'  # @param {type:'string'}
DIRECTORIO_ESTUDIO = '/content/drive/MyDrive/Replica_Toprani_2025'  # @param {type:'string'}
DIRECTORIO_EVALUADOR = '/content/drive/MyDrive/Evaluador_Toprani'  # @param {type:'string'}
CONFIRMAR_REVISION_FORMAL = False  # @param {type:'boolean'}
PROBAR_API = True  # @param {type:'boolean'}

if MODO not in {'simular', 'detectar', 'evaluar'}:
    raise ValueError('Elige un modo definido.')
drive.mount('/content/drive')
root = Path(DIRECTORIO_ESTUDIO)
private_root = Path(DIRECTORIO_EVALUADOR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
# Archivos públicos congelados del estudio, sin etiquetas.
paquete = base64.b64decode('...', validate=True)
assert hashlib.sha256(paquete).hexdigest() == '2425c4a85ac8d61bd7d1be1357c8c0d5c018ffdd41841f6df96dadb84116d717'
with zipfile.ZipFile(io.BytesIO(paquete)) as archive:
    for entry in archive.infolist():
        destination = root / entry.filename
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(entry))

requerido = dict(line.split('==') for line in (root / 'requirements.txt').read_text().splitlines() if line)
def version_instalada(name):
    from importlib.metadata import version, PackageNotFoundError
    try:
        return version(name)
    except PackageNotFoundError:
        return None

if any(version_instalada(name) != version for name, version in requerido.items()):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')])
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != requerido['numpy']:
    raise RuntimeError('Reinicia la sesión una vez para cargar la versión de NumPy instalada.')
sys.path.insert(0, str(root))
import pipeline as study
study = importlib.reload(study)
config = study.load_config(root)
print('Carpeta del estudio:', root)

Carpeta del estudio: /content/drive/MyDrive/Replica_Toprani_2025


## 2. Entradas y normalización

Las plantillas YAML o JSON se conservan junto a un resumen con recursos, parámetros efectivos, condiciones y referencias. Un parámetro externo se documenta en el contexto del caso. No se expande un macro ni se consulta el estado de una pila.

In [78]:
manifest = study.suite_cases(root, FASE)
case = manifest[0]
corpus = root / 'corpus' / FASE
sample = corpus / case['archivo']
preview = study.normalize(sample.read_text(encoding='utf-8'), sample.suffix,
                          study.read_json(corpus / case['contexto']))
print('Casos:', ', '.join(row['plantilla_id'] for row in manifest))
print('Tipos en el primer caso:', ', '.join(item['Type'] for item in preview['recursos']))

Casos: T01, T02, T03, T04, T05, T06, T07, T08, T09, T10
Tipos en el primer caso: AWS::S3::Bucket, AWS::SecretsManager::Secret, AWS::RDS::DBInstance, AWS::IAM::Role, AWS::IAM::Policy


## 3. Documentación y versiones

Las instantáneas contienen paráfrasis de documentación oficial de AWS con URL y huella. `actualizada` añade el anuncio sobre Secrets Manager. El benchmark CIS completo requiere acceso al material oficial y no se incluye en la base preconfigurada. Puedes versionar documentación adicional antes de cerrar la referencia formal.

In [79]:
base_docs = study.docs_for_version(root, 'base')
new_docs = study.docs_for_version(root, 'actualizada')
assert [row for row in new_docs if not row['fragmento_id'].startswith('AVISO-')] == base_docs
print('Fragmentos en base:', len(base_docs), '· Con actualización:', len(new_docs))

Fragmentos en base: 11 · Con actualización: 12


## 4. Embeddings y prueba de Bedrock

Titan V2 devuelve los vectores. El índice NumPy se conserva en Drive con sus huellas. La prueba de acceso comprueba el embedding y calienta los dos esquemas JSON antes de medir la corrida. Una credencial vencida se renueva en Colab Secrets.

In [85]:
if MODO == 'simular':
    model = study.Simulation(config)

elif MODO == 'detectar':
    from importlib.metadata import PackageNotFoundError, version

    sdk_version = '3.5.0'
    try:
        instalado = version('google-genai')
    except PackageNotFoundError:
        instalado = None

    if instalado != sdk_version:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            f'google-genai=={sdk_version}'
        ])

    from google import genai
    from google.colab import userdata
    from google.genai import types

    config.update(
        proveedor='gemini',
        text_model='gemini-3.8-flash',
        embedding_model='gemini-embedding-2',
        dimensions=768,
        region=None,
        temperature=None,
        thinking_level='LOW',
        google_genai_version=sdk_version,
    )
    study.save_json(root / 'config/config.yaml', config)

    class Gemini:
        def __init__(self, config, token):
            self.config = config
            self.client = genai.Client(api_key=token)

        def embed(self, text, ledger, stage):
            model_id = self.config['embedding_model']
            response = self.client.models.embed_content(
                model=model_id,
                contents=text,
                config={'output_dimensionality': self.config['dimensions']},
            )
            ledger({'etapa': stage, 'modelo': model_id, 'uso': {}})
            return response.embeddings[0].values

        def generate(self, prompt, payload, schema, stage, ledger):
            model_id = self.config['text_model']
            response = self.client.models.generate_content(
                model=model_id,
                contents=study.encode(payload).decode('utf-8'),
                config={
                    'system_instruction': prompt,
                    'response_mime_type': 'application/json',
                    'response_json_schema': schema,
                    'max_output_tokens': self.config[f'{stage}_max_tokens'],
                    'thinking_config': {
                        'thinking_level': self.config['thinking_level']
                    },
                },
            )
            usage = response.usage_metadata
            ledger({
                'etapa': stage,
                'modelo': model_id,
                'uso': {
                    'inputTokens': getattr(usage, 'prompt_token_count', 0) or 0,
                    'outputTokens': (
                        (getattr(usage, 'candidates_token_count', 0) or 0)
                        + (getattr(usage, 'thoughts_token_count', 0) or 0)
                    ),
                },
            })
            candidate = (response.candidates or [None])[0]
            reason = getattr(candidate, 'finish_reason', None)
            return {
                'texto': response.text or '',
                'stop_reason': (
                    'end_turn' if reason == types.FinishReason.STOP else str(reason)
                ),
            }

    key = userdata.get('GEMINI_API_KEY')
    if not key:
        raise ValueError('Agrega GEMINI_API_KEY y habilita el acceso del cuaderno al Secret.')
    model = Gemini(config, token=key)
    del key
    print('Gemini configurado exitosamente utilizando la API Key provista.')

In [81]:
def registrar(evento):
    pass

vector = model.embed("Prueba de conexión", registrar, "prueba")
respuesta = model.generate(
    "Devuelve una lista vacía de hallazgos.",
    {}, study.DETECTOR, "detector", registrar,
)

assert len(vector) == config["dimensions"]
assert study.parse_response(respuesta, study.DETECTOR) == {"hallazgos": []}
print(f"Estado: OK. Gemini operativo: embedding de {len(vector)} dimensiones y JSON válido.")

Estado: OK. Gemini operativo: embedding de 768 dimensiones y JSON válido.


## 5–7. Recuperación, detector e informe

La recuperación selecciona fragmentos por similitud coseno. El detector recibe la plantilla original, su resumen y los fragmentos. El generador recibe los hallazgos congelados y propone acciones en Markdown. Reabrir el mismo ID reutiliza etapas guardadas que superen la verificación de archivos.

`formal` ejecuta tres repeticiones. `actualizacion` ejecuta tres pares sobre ambas versiones documentales y alterna el orden. Inicia estas fases después de revisar el piloto y estimar su consumo. La fase formal real requiere el cierre de referencia aprobado por el especialista.

In [82]:
def nombres_de_corrida(fase, prefix):
    if fase == 'actualizacion':
        for index in range(1, 4):
            arms = ('base', 'actualizada') if index % 2 else ('actualizada', 'base')
            for variant in arms:
                yield f'{prefix}-r{index}-{variant}', variant
    elif fase == 'formal':
        for index in range(1, 4):
            yield f'{prefix}-r{index}', 'base'
    else:
        yield prefix, 'base'

if MODO in {'simular', 'detectar'}:
    for name, version in nombres_de_corrida(FASE, ID_CORRIDA):
        run, progress = study.execute(root, FASE, name, version, model, simulation=MODO == 'simular')
        print(name, [(item['plantilla_id'], item['estado']) for item in progress])
        incomplete = [item for item in progress if item['estado'] != 'completa']
        if incomplete:
            print('Corrida incompleta. Consulta progreso.json y reanuda con el mismo ID.')
            print(incomplete)
            break

## 8. Puntuación y revisión humana

En la carpeta privada del evaluador pon `Referencias_Toprani_Evaluador.zip` o las carpetas extraídas `smoke/`, `piloto/`, `formal/` y `actualizacion/`. Usa `MODO = "evaluar"` y el ID y la fase que ya ejecutaste.

Para **cerrar el corpus antes de la corrida formal**, el especialista revisa `formal/referencia.csv`, completa `revisor` y `fecha_revision` (`AAAA-MM-DD`) en cada fila, elige `FASE = "formal"` y activa `CONFIRMAR_REVISION_FORMAL`. Se escribe una constancia compartida que contiene solo la huella de las etiquetas. Desactiva la opción para puntuar después de la ejecución.

La rúbrica `remediacion.csv` usa 0 para un criterio incumplido, 1 si se cumple parcialmente y 2 si se cumple. El especialista revisa corrección, especificidad, respaldo y contexto. Marca sobre-remediación, citas incorrectas y propiedades obsoletas.

In [83]:
if MODO == 'evaluar':
    reference = private_root / FASE / 'referencia.csv'
    private_zip = private_root / 'Referencias_Toprani_Evaluador.zip'
    if not reference.exists() and private_zip.exists():
        with zipfile.ZipFile(private_zip) as archive:
            for phase in ('smoke', 'piloto', 'formal', 'actualizacion'):
                path = private_root / phase / 'referencia.csv'
                if not path.exists():
                    study.save(path, archive.read(f'{phase}/referencia.csv'))
    if not reference.exists():
        raise FileNotFoundError(f'Falta la referencia privada: {reference}')
    if CONFIRMAR_REVISION_FORMAL:
        if FASE != 'formal':
            raise ValueError('Selecciona la fase formal para cerrar la referencia.')
        study.review_formal(root, reference, confirmed=True)
        print('Referencia cerrada. Resultado: config/formal_lock.json')
    else:
        for name, _ in nombres_de_corrida(FASE, ID_CORRIDA):
            run = root / 'runs' / name
            metrics = study.evaluate(root, run, reference)
            print(name, f"{metrics['completas']}/{metrics['programadas']}", metrics['micro'])
        if FASE == 'formal':
            print(study.formal_summary(root, ID_CORRIDA))
        if FASE == 'actualizacion':
            print('Filas en la comparación pareada:', len(study.update_summary(root, ID_CORRIDA)))

formal-gemini-r1 10/10 {'TP': 20, 'FP': 0, 'FN': 0, 'duplicado': 0, 'abstencion': 0, 'indeterminado': 0, 'fuera_catalogo': 0, 'cita_invalida': 0, 'ruta_invalida': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'fraccion_alertas_incorrectas': 0.0}
formal-gemini-r2 10/10 {'TP': 20, 'FP': 0, 'FN': 0, 'duplicado': 0, 'abstencion': 0, 'indeterminado': 0, 'fuera_catalogo': 0, 'cita_invalida': 0, 'ruta_invalida': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'fraccion_alertas_incorrectas': 0.0}
formal-gemini-r3 10/10 {'TP': 20, 'FP': 0, 'FN': 0, 'duplicado': 0, 'abstencion': 0, 'indeterminado': 0, 'fuera_catalogo': 0, 'cita_invalida': 0, 'ruta_invalida': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'fraccion_alertas_incorrectas': 0.0}
{'simulacion': False, 'original_Toprani_Madisetti': {'TP': 17, 'FP': 3, 'FN': 3, 'precision': 0.85, 'recall': 0.85, 'f1': 0.85, 'recuperacion_s': 2.07, 'detector_s': 70, 'informe_s': 13, 'total_s': 85.1}, 'replica': {'TP': {'mediana': 20, 'min': 20, 'max': 20

## Consulta de resultados y presupuesto del piloto

El informe puntuado de cada plantilla está en `runs/<ID>/<plantilla>/informe_puntuado.md`. El consumo se basa en las respuestas de la API. Introduce tarifas regionales por millón de tokens si deseas estimar dólares. El presupuesto base cubre treinta análisis formales y doce del ensayo documental.

In [ ]:
CALCULAR_PRESUPUESTO = False  # @param {type:'boolean'}
TARIFAS = None  # Opcional. {id_de_modelo: {'entrada': precio_por_millon, 'salida': precio_por_millon}}
if CALCULAR_PRESUPUESTO:
    budget = study.estimate_usage(root / 'runs' / ID_CORRIDA, root=root, rates=TARIFAS)
    study.save_json(root / 'runs' / ID_CORRIDA / 'presupuesto.json', budget)
    print(budget)